# 04 — Claude Portfolio Analysis

This notebook feeds your enriched portfolio data into Claude and gets back structured
**BUY / SELL / HOLD** recommendations for every position.

Claude acts as a senior financial analyst, weighing technicals, fundamentals, performance,
and recent news to produce actionable (but advisory-only) output.

**Prerequisites:**
- Run notebooks 01–03 first
- `enriched_portfolio.json` must exist
- `ANTHROPIC_API_KEY` in your `.env` file (get one at https://console.anthropic.com/)
- Run `uv sync` after pulling this update (new dependency: anthropic)

## Setup

In [1]:
import json
import os
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(os.path.join("..", ".env"), override=True)

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError(
        "Missing ANTHROPIC_API_KEY in .env. "
        "Get one at https://console.anthropic.com/"
    )

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Load enriched portfolio
ENRICHED_FILE = os.path.join("..", "enriched_portfolio.json")

if not os.path.exists(ENRICHED_FILE):
    raise FileNotFoundError(
        "enriched_portfolio.json not found. Run notebook 03 first."
    )

with open(ENRICHED_FILE, "r") as f:
    enriched = json.load(f)

print(f"✅ Claude client initialized")
print(f"✅ Loaded enriched portfolio: {len(enriched['holdings'])} positions, {len(enriched['enrichment'])} enriched tickers")
print(f"   Generated at: {enriched['generated_at']}")

✅ Claude client initialized
✅ Loaded enriched portfolio: 44 positions, 36 enriched tickers
   Generated at: 2026-03-20T16:26:12.964877+00:00


In [2]:
# Preflight: verify API key and model access with a minimal call
MODEL = "claude-sonnet-4-6"

print(f"🔍 Running API preflight check (model: {MODEL})...")
try:
    _test = claude.messages.create(
        model=MODEL,
        max_tokens=5,
        messages=[{"role": "user", "content": "ping"}]
    )
    print(f"✅ API preflight OK — credentials and model access confirmed")
except anthropic.AuthenticationError as e:
    print(f"❌ API key invalid or expired: {e}")
    raise
except anthropic.PermissionDeniedError as e:
    print(f"❌ Model not available on this plan: {e}")
    raise
except anthropic.BadRequestError as e:
    if "credit" in str(e).lower():
        print(f"❌ Insufficient credits — top up at: https://console.anthropic.com/settings/plans")
        print(f"   Details: {e}")
    else:
        print(f"❌ Bad request during preflight: {e}")
    raise
except Exception as e:
    print(f"❌ Preflight failed ({type(e).__name__}): {e}")
    raise

🔍 Running API preflight check (model: claude-sonnet-4-6)...


✅ API preflight OK — credentials and model access confirmed


## System Prompt — Financial Analyst Persona

In [3]:
SYSTEM_PROMPT = """You are a senior financial analyst conducting a portfolio review.

You will receive a JSON object containing:
- "summary": portfolio-level totals (value, cost basis, gain/loss)
- "holdings": array of positions (ticker, brokerage, quantity, cost_basis, value, account_type, account_subtype)
- "enrichment": per-ticker object with technicals, fundamentals, performance, and news
- "recent_transactions" (optional): up to 50 investment transactions from the last 30 days

Your job is to analyze every non-cash position and produce a structured recommendation.

## Analysis Framework
For each position, evaluate:
1. **Technical signals** — RSI (overbought >70, oversold <30), SMA trend (golden/death cross), MACD momentum, Bollinger Band position
2. **Fundamental quality** — P/E vs sector, revenue/earnings growth, margins, debt levels, analyst consensus
3. **Performance context** — recent returns (1d/1w/1m/3m), proximity to 52-week high/low, analyst target upside
4. **News sentiment** — any material headlines (earnings, lawsuits, guidance changes, macro events)
5. **Portfolio context** — concentration risk, sector overlap, position sizing
6. **Recent activity** — if recent_transactions is provided, note recent buys/sells and whether they align with your recommendations. Flag if the user recently bought something you'd recommend selling, or vice versa.

## Account Type Awareness
Each holding includes "account_type": "taxable" | "retirement" | "liquid" | "other"
and "account_subtype" (raw Plaid value: "ira", "roth", "brokerage", "savings", etc.)

## Rules by Account Type
- **taxable**: Full BUY/SELL/HOLD analysis with tax-loss harvesting context. Include in "recommendations".
- **retirement**: Full BUY/SELL/HOLD analysis with a long term view included in "retirement_summary" section only. These are long-term buy-and-hold — provide growth notes only, no action calls. Note Traditional IRA vs Roth IRA distinction where relevant for tax context. Or conviction bets needing an appropriate stop loss. Options are likely leveraged long term bets which are candidates for rolling.
- **liquid**: Skip entirely — do not analyze cash or savings positions.
- If a ticker appears in BOTH taxable AND retirement accounts, analyze the taxable holding normally in "recommendations" and note the retirement holding separately in "retirement_summary".

## Output Format
Respond with ONLY a valid JSON object (no markdown fences, no preamble) in this exact structure:

{
  "analysis_date": "YYYY-MM-DD",
  "portfolio_assessment": {
    "overall_health": "strong | moderate | weak",
    "summary": "2-3 sentence portfolio-level assessment",
    "sector_concentration": "note any over-concentration",
    "risk_level": "low | moderate | high",
    "top_concern": "single biggest risk or issue"
  },
  "recommendations": [
    {
      "ticker": "AAPL",
      "name": "Apple Inc.",
      "brokerage": "stash",
      "action": "BUY | SELL | HOLD | ROLL",
      "confidence": "high | medium | low",
      "urgency": "immediate | soon | no_rush",
      "thesis": "2-3 sentence rationale combining technicals + fundamentals + news",
      "bull_case": "1 sentence best case scenario",
      "bear_case": "1 sentence worst case scenario",
      "key_signals": ["RSI 32 oversold", "golden cross forming", "strong earnings beat"],
      "risk_factors": ["high P/E", "sector rotation risk"],
      "position_note": "optional note on sizing — trim, add, or hold current size"
    }
  ],
  "retirement_summary": {
    "total_value": 40000,
    "total_positions": 15,
    "assessment": "2-3 sentence assessment of retirement allocation overall — diversification, growth trajectory, any notable risks.",
    "holdings": [
      {
        "ticker": "JEPQ",
        "brokerage": "robinhood",
        "account_subtype": "ira",
        "value": 11210,
        "growth_note": "1 sentence growth/quality note — no buy/sell/hold"
      }
    ]
  },
  "watchlist": [
    {
      "ticker": "MSFT",
      "reason": "approaching oversold territory, watch for entry"
    }
  ],
  "action_items": [
    "Highest priority action in plain English",
    "Second priority action",
    "Third priority action"
  ]
}

## Rules
- Every non-cash **taxable** holding MUST get a recommendation
- **Retirement** holdings go ONLY in retirement_summary — never in recommendations
- **Liquid** (cash/savings) holdings are skipped entirely
- Be specific with numbers — reference actual RSI values, P/E ratios, return percentages
- If data is missing for a ticker, note it and give best available assessment
- Watchlist is for positions NOT currently held that are worth monitoring
- Action items should be the 3-5 most important things to do right now
- **ROLL** is only valid for options positions — use it when the best action is to roll the contract forward rather than close. Include roll instructions (new expiry, strike) in "thesis".
- Be direct — this user wants clear calls, not hedging
- If recent_transactions is present: note in action_items or position_note if the user's recent trades conflict with your recommendations

## Disclaimer
You are providing analysis for informational purposes only. This is not financial advice.
The user makes all final investment decisions.


IMPORTANT OUTPUT CONSTRAINTS:
- Keep each recommendation's "thesis" to exactly 1-2 sentences. No filler.
- "bull_case" and "bear_case": one sentence each, max.
- "key_signals": 3-5 bullet items max per position.
- "risk_factors": 2-3 items max per position.
- Skip positions with no meaningful signal (e.g. cash sweeps, money market funds) — just omit them.
- Prioritize depth on your highest-conviction calls. Low-conviction HOLDs can be brief.
- Total response must fit within 4000 words.

## International events and markets ##
In the light of the recent events around the world, you are aware that the markets are volatile and the situation is uncertain.
Provide the analysis in the context of the current geopolitical situation. If there are critical events like stock corrections or market crashes, mention it in the analysis and
be sure to pin this to the top of the analysis. Help the user save money and protect their portfolio."""

print(f"✅ System prompt ready ({len(SYSTEM_PROMPT)} chars)")

✅ System prompt ready (6179 chars)


## Prepare the Payload

We trim the data to stay well within Claude's context window while keeping everything material.

In [4]:
# Build a focused payload — keep everything Claude needs, trim noise
payload = {
    "summary": enriched["summary"],
    "holdings": [],
    "enrichment": {},
}

for h in enriched["holdings"]:
    # Include all holdings so Claude sees the full picture
    payload["holdings"].append({
        "ticker": h.get("ticker"),
        "name": h.get("name"),
        "brokerage": h.get("brokerage"),
        "type": h.get("type"),
        "account_type": h.get("account_type"),
        "account_subtype": h.get("account_subtype"),
        "quantity": h.get("quantity"),
        "cost_basis": h.get("cost_basis"),
        "price": h.get("price"),
        "value": h.get("value"),
        "gain_loss": h.get("gain_loss"),
        "gain_loss_pct": h.get("gain_loss_pct"),
    })

# Include all enrichment data
for ticker, data in enriched["enrichment"].items():
    enrichment_entry = {}
    
    # Technicals — keep everything
    if data.get("technicals"):
        enrichment_entry["technicals"] = data["technicals"]
    
    # Fundamentals — keep everything except long description
    if data.get("fundamentals"):
        fund = {k: v for k, v in data["fundamentals"].items() if k != "short_description"}
        enrichment_entry["fundamentals"] = fund
    
    # Performance — keep everything
    if data.get("performance"):
        enrichment_entry["performance"] = data["performance"]
    
    # News — keep titles and publishers only (saves tokens)
    if data.get("news"):
        enrichment_entry["news"] = [
            {"title": n.get("title"), "publisher": n.get("publisher"), "date": n.get("published")}
            for n in data["news"]
        ]
    
    payload["enrichment"][ticker] = enrichment_entry

# ── Include recent transactions (last 30 days, capped at 50) ──────────────
from datetime import date, timedelta
import os

TRANSACTIONS_FILE = os.path.join("..", "transactions.json")
if os.path.exists(TRANSACTIONS_FILE):
    with open(TRANSACTIONS_FILE) as f:
        all_txns = json.load(f)
    
    cutoff = (date.today() - timedelta(days=30)).isoformat()
    recent_txns = [
        {
            "date": t["date"],
            "brokerage": t["brokerage"],
            "ticker": t["ticker"],
            "type": t["type"],
            "quantity": t["quantity"],
            "price": t["price"],
            "amount": t["amount"],
        }
        for t in all_txns
        if t.get("date", "") >= cutoff
    ]
    # Cap at 50 to save tokens
    recent_txns = sorted(recent_txns, key=lambda x: x["date"], reverse=True)[:50]
    if recent_txns:
        payload["recent_transactions"] = recent_txns
        print(f"   ✅ Included {len(recent_txns)} transactions from last 30 days")
    else:
        print("   ℹ️  No transactions in last 30 days")
else:
    print("   ℹ️  transactions.json not found — transaction context omitted")

payload_json = json.dumps(payload, indent=2, default=str)
payload_tokens_est = len(payload_json) // 4  # rough estimate

print(f"📦 Payload ready:")
print(f"   {len(payload['holdings'])} holdings")
print(f"   {len(payload['enrichment'])} enriched tickers")
print(f"   {len(payload.get('recent_transactions', []))} recent transactions")
print(f"   ~{payload_tokens_est:,} tokens (estimated)")
print(f"   {len(payload_json):,} chars")

   ℹ️  transactions.json not found — transaction context omitted
📦 Payload ready:
   44 holdings
   36 enriched tickers
   0 recent transactions
   ~20,553 tokens (estimated)
   82,212 chars


import httpx

try:
    message = claude.messages.create(
        model=MODEL,
        max_tokens=32768,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": f"Here is my current portfolio data as of {datetime.now().strftime('%Y-%m-%d %H:%M')}.\n"
                           f"Please analyze every position and provide your structured recommendations.\n\n"
                           f"{payload_json}"
            }
        ],
        timeout=httpx.Timeout(600.0, connect=5.0),
    )
    
    raw_response = message.content[0].text

    print(f"✅ Response received")
    print(f"   Model: {message.model}")
    print(f"   Input tokens: {message.usage.input_tokens:,}")
    print(f"   Output tokens: {message.usage.output_tokens:,}")
    print(f"   Stop reason: {message.stop_reason}")

except anthropic.BadRequestError as e:
    if "credit balance" in str(e).lower() or "credit" in str(e).lower():
        print(f"\n❌ Anthropic API Error: Credit balance too low.")
        print(f"   Please top up your credits at: https://console.anthropic.com/settings/plans")
        print(f"   Detailed error: {e}")
    else:
        print(f"\n❌ Anthropic API Bad Request: {e}")
    raw_response = None
    raise e
except anthropic.RateLimitError as e:
    print(f"\n❌ Anthropic API Rate Limit: {e}")
    raw_response = None
    raise e
except httpx.TimeoutException as e:
    print(f"\n❌ Anthropic API Timeout: The request took too long. Try again or use a smaller model.")
    raw_response = None
    raise e
except Exception as e:
    print(f"\n❌ Unexpected Error ({type(e).__name__}): {e}")
    raw_response = None
    raise e

In [5]:
import httpx
# Choose your model
MODEL = "claude-sonnet-4-6"  # New sonnet model that's great for financial analysis

try:
    message = claude.messages.create(
        model=MODEL,
        max_tokens=32768,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": f"Here is my current portfolio data as of {datetime.now().strftime('%Y-%m-%d %H:%M')}.\n"
                           f"Please analyze every position and provide your structured recommendations.\n\n"
                           f"{payload_json}"
            }
        ],
        timeout=httpx.Timeout(600.0, connect=5.0),
    )
    
    raw_response = message.content[0].text

    print(f"✅ Response received")
    print(f"   Model: {message.model}")
    print(f"   Input tokens: {message.usage.input_tokens:,}")
    print(f"   Output tokens: {message.usage.output_tokens:,}")
    print(f"   Stop reason: {message.stop_reason}")

except anthropic.BadRequestError as e:
    if "credit balance" in str(e).lower():
        print(f"\n❌ Anthropic API Error: Credit balance too low.")
        print(f"   Please top up your credits at: https://console.anthropic.com/settings/plans")
        print(f"   Detailed error: {e}")
    else:
        print(f"\n❌ Anthropic API Bad Request: {e}")
    raw_response = None
    raise e
except anthropic.RateLimitError as e:
    print(f"\n❌ Anthropic API Rate Limit: {e}")
    raw_response = None
    raise e
except httpx.TimeoutException:
    print(f"\n❌ Anthropic API Timeout: The request took too long. Try again or use a smaller model.")
    raw_response = None
    raise e
except Exception as e:
    print(f"\n❌ Unexpected Error: {e}")
    raw_response = None
    raise e


✅ Response received
   Model: claude-sonnet-4-6
   Input tokens: 32,440
   Output tokens: 7,242
   Stop reason: end_turn


## Parse Recommendations

In [6]:
if raw_response is None:
    print("⚠️ Skipping analysis parsing because raw_response is None. (API error)")
    analysis = None
else:
    # Clean response (strip markdown fences if Claude included them despite instructions)
    cleaned = raw_response.strip()
    if cleaned.startswith("```"):
        # Remove opening fence
        cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].strip()

    try:
        analysis = json.loads(cleaned)
        print("✅ Successfully parsed Claude's analysis as JSON")
    except json.JSONDecodeError as e:
        print(f"⚠️  JSON parse error: {e}")
        print(f"\nRaw response (first 500 chars):")
        print(raw_response[:500])
        print("\n💡 You can try re-running the analysis cell, or use the raw response below.")
        analysis = None


✅ Successfully parsed Claude's analysis as JSON


## Portfolio Assessment

In [7]:
if analysis and "portfolio_assessment" in analysis:
    pa = analysis["portfolio_assessment"]
    
    health_emoji = {"strong": "🟢", "moderate": "🟡", "weak": "🔴"}.get(pa.get("overall_health"), "⚪")
    risk_emoji = {"low": "🟢", "moderate": "🟡", "high": "🔴"}.get(pa.get("risk_level"), "⚪")
    
    print("═" * 60)
    print("         PORTFOLIO ASSESSMENT")
    print("═" * 60)
    print(f"  Health:     {health_emoji} {pa.get('overall_health', 'N/A').upper()}")
    print(f"  Risk Level: {risk_emoji} {pa.get('risk_level', 'N/A').upper()}")
    print(f"")
    print(f"  Summary:")
    print(f"  {pa.get('summary', 'N/A')}")
    print(f"")
    print(f"  Sector Concentration:")
    print(f"  {pa.get('sector_concentration', 'N/A')}")
    print(f"")
    print(f"  ⚠️  Top Concern:")
    print(f"  {pa.get('top_concern', 'N/A')}")
    print("═" * 60)
else:
    print("⚠️  No portfolio assessment found in response.")

════════════════════════════════════════════════════════════
         PORTFOLIO ASSESSMENT
════════════════════════════════════════════════════════════
  Health:     🔴 WEAK
  Risk Level: 🔴 HIGH

  Summary:
  The portfolio is broadly under pressure — only $1,772 in total gain on $87K cost basis (2% gain) masks significant losing positions offset by a few large winners (META +266%, TSLA +79%). The majority of holdings are trading below their 50-day SMAs, multiple RSI readings are oversold or approaching it, and the macro/market environment in early 2026 reflects continued volatility with broad equity drawdowns, trade war concerns, and risk-off rotation. The options book is a particular drag, with several LEAPS deeply underwater.

  Sector Concentration:
  Heavy Real Estate/REIT exposure in taxable (O, AHRT, VICI = ~12% of taxable). Healthcare overweight across both taxable (NVO call) and retirement (MRK, UNH, PFE, SNAP calls). China exposure concentrated in JD calls and BABA. Multiple sp

## Position Recommendations

In [8]:
if analysis and "recommendations" in analysis:
    recs = analysis["recommendations"]
    
    # Summary table
    rec_rows = []
    for r in recs:
        action_emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "🟡", "ROLL": "🔵"}.get(r.get("action"), "⚪")
        rec_rows.append({
            "action": f"{action_emoji} {r.get('action', 'N/A')}",
            "ticker": r.get("ticker"),
            "name": r.get("name", "")[:30],
            "brokerage": r.get("brokerage"),
            "confidence": r.get("confidence"),
            "urgency": r.get("urgency"),
            "position_note": r.get("position_note", ""),
        })
    
    rec_df = pd.DataFrame(rec_rows)
    
    # Count by action
    buys = sum(1 for r in recs if r.get("action") == "BUY")
    sells = sum(1 for r in recs if r.get("action") == "SELL")
    holds = sum(1 for r in recs if r.get("action") == "HOLD")
    rolls = sum(1 for r in recs if r.get("action") == "ROLL")
    
    print(f"📊 Recommendations: {buys} BUY | {holds} HOLD | {sells} SELL | {rolls} ROLL\n")
    
    rec_df

📊 Recommendations: 1 BUY | 8 HOLD | 4 SELL | 1 ROLL



## Detailed Analysis Per Position

In [9]:
if analysis and "recommendations" in analysis:
    for i, r in enumerate(analysis["recommendations"]):
        action_emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "🟡", "ROLL": "🔵"}.get(r.get("action"), "⚪")
        conf_emoji = {"high": "🔵", "medium": "🟡", "low": "⚪"}.get(r.get("confidence"), "⚪")
        
        print(f"{'─' * 60}")
        print(f"{action_emoji} {r.get('action', 'N/A')}  │  {r.get('ticker', '?')} — {r.get('name', '')}")
        print(f"   Confidence: {conf_emoji} {r.get('confidence', 'N/A')}  │  Urgency: {r.get('urgency', 'N/A')}  │  Brokerage: {r.get('brokerage', 'N/A')}")
        print(f"")
        print(f"   📝 {r.get('thesis', 'No thesis provided')}")
        print(f"")
        print(f"   📈 Bull: {r.get('bull_case', 'N/A')}")
        print(f"   📉 Bear: {r.get('bear_case', 'N/A')}")
        print(f"")
        
        signals = r.get("key_signals", [])
        if signals:
            print(f"   Key Signals:")
            for s in signals:
                print(f"     • {s}")
        
        risks = r.get("risk_factors", [])
        if risks:
            print(f"   Risk Factors:")
            for rf in risks:
                print(f"     ⚠️  {rf}")
        
        if r.get("position_note"):
            print(f"\n   💼 Position: {r['position_note']}")
        print()

────────────────────────────────────────────────────────────
🟡 HOLD  │  META — Meta Platforms Inc.
   Confidence: 🔵 high  │  Urgency: no_rush  │  Brokerage: Fidelity

   📝 META is down 25% from 52-week highs with RSI at 33 and price breaking below the lower Bollinger Band — technically oversold. Fundamentals remain elite: 30% profit margins, 24% revenue growth, forward P/E of 16.6x, and analyst consensus strong buy with a $864 target implying 45% upside.

   📈 Bull: AI-driven ad revenue acceleration and cost discipline push META back toward $800 within 12 months.
   📉 Bear: Regulatory crackdowns or ad market slowdown compress multiples further in a risk-off environment.

   Key Signals:
     • RSI 33.2 approaching oversold
     • Price below lower Bollinger Band
     • Forward P/E 16.6x is cheap for quality
     • 45% analyst upside to $864 target
   Risk Factors:
     ⚠️  Macro risk-off punishing high-beta growth
     ⚠️  Continued technical downtrend — no base yet

   💼 Position: Hol

## Watchlist

In [10]:
if analysis and analysis.get("watchlist"):
    print("👀 WATCHLIST — Not currently held, but worth monitoring:\n")
    for w in analysis["watchlist"]:
        print(f"  📌 {w.get('ticker', '?')}: {w.get('reason', '')}")
else:
    print("No watchlist items.")

👀 WATCHLIST — Not currently held, but worth monitoring:

  📌 SNOW: Snowflake shows RSI 44 stabilizing with 30% revenue growth and strong buy consensus; down 39% from 52-week high — watch for a technical base formation around $160-165 before initiating a position
  📌 ADBE: ADBE stock (not just the calls) is worth watching — if it stabilizes above $280 the Roth call spread recovers significantly; a base at current levels would be a potential entry
  📌 LLY: Eli Lilly is the GLP-1 market leader taking share from NVO — as you exit NVO, LLY on a pullback could be a better healthcare replacement with stronger fundamentals


## Action Items — What to Do Right Now

In [11]:
if analysis and analysis.get("action_items"):
    print("🎯 PRIORITY ACTION ITEMS:\n")
    for i, item in enumerate(analysis["action_items"], 1):
        print(f"  {i}. {item}")
else:
    print("No action items found.")

🎯 PRIORITY ACTION ITEMS:

  1. URGENT — Close or roll the PALL Jun 2026 $135 call in the Roth IRA immediately: it expires in ~3 months, is down 52%, and palladium would need to surge dramatically to recover; rolling to a Jan 2027 lower strike or closing now salvages $1,630 before time decay destroys remaining value
  2. Sell NVO shares (taxable) and NVO $30 call simultaneously to harvest ~$2,256 in combined tax losses — ensure you wait 31 days before repurchasing NVO in any account to avoid wash sale disallowance
  3. Sell AHRT (taxable) and harvest the -$251 tax loss; redeploy proceeds into VICI which is higher quality, equally oversold at RSI 26.9, and offers better fundamentals at its 52-week low
  4. Roll the 3x JD Jan 2027 $25 calls into JD Jun 2027 $25 calls to consolidate with your existing Jun 2027 position and extend duration on the China recovery thesis
  5. Sell LULU (taxable) to harvest the -$681 tax loss — combined with NVO and AHRT, total taxable loss harvesting available

## Save Analysis Report

In [12]:
if analysis:
    report = {
        "generated_at": datetime.now().isoformat(),
        "model": message.model,
        "usage": {
            "input_tokens": message.usage.input_tokens,
            "output_tokens": message.usage.output_tokens,
        },
        "analysis": analysis,
    }
    
    # Save with timestamp for history
    reports_dir = os.path.join("..", "reports")
    os.makedirs(reports_dir, exist_ok=True)
    
    date_str = datetime.now().strftime("%Y-%m-%d_%H%M")
    report_path = os.path.join(reports_dir, f"analysis_{date_str}.json")
    
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2, default=str)
    
    # Also save as "latest" for easy access
    latest_path = os.path.join(reports_dir, "latest_analysis.json")
    with open(latest_path, "w") as f:
        json.dump(report, f, indent=2, default=str)
    
    print(f"💾 Analysis saved:")
    print(f"   {report_path}")
    print(f"   {latest_path}")
    print(f"\n📌 These reports will be used for performance tracking in Phase 5.")
else:
    print("⚠️  No analysis to save. Check the parsing step above.")

💾 Analysis saved:
   ../reports/analysis_2026-03-20_2158.json
   ../reports/latest_analysis.json

📌 These reports will be used for performance tracking in Phase 5.


## Raw Response (Debug)

If the JSON parsing failed, you can inspect Claude's raw output here.

In [13]:
# Uncomment to see raw response:
# print(raw_response)

---

### ⚠️ Disclaimer
This analysis is generated by an AI for informational purposes only. It is **not financial advice**.
Always do your own research and consult a qualified financial advisor before making investment decisions.

### Next Steps
- **Phase 4 (notebook 05)**: Automated notifications — get these recommendations emailed/pushed to you on a schedule
- **Phase 5**: Performance tracking — log recommendations and measure accuracy over time